# Obesity ML pipeline — visualization (COMP315)

End-to-end **MLOps** artifacts from the **TFX** pipeline (see `obesity_tfx/tfx_pipeline.py`), orchestrated by **Airflow** (`airflow_dags/obesity_dag.py`). This notebook **inspects and visualizes** what each stage produced — it does not re-train the model.

| Pipeline step | Component | Covered here |
|---|---|---|
| 1 | `CsvExampleGen` | Step 1 — train/eval TFRecord paths |
| 2 | `StatisticsGen` | Step 2 — TFDV train vs eval stats |
| 3 | `SchemaGen` | Step 3 — inferred schema |
| 4 | `ExampleValidator` | Step 4 — anomalies vs schema |
| 5 | `Transform` (TFT) | Step 5 — post-transform statistics |
| 6 | `Trainer` | **Step 6** — SavedModel (`Format-Serving`) |
| 8 | `Evaluator` (TFMA) | Step 8 — metrics, slices, time series, plots |
| 9 | `Pusher` | **Step 9** — blessed model push path |

**Prerequisite:** run the **`obesity_ml_pipeline`** DAG so artifacts exist under `tfx_airflow_runs/outputs/` (or the Airflow copy under `.../dags/COMP315_Group8_project/`).

**Environment:** **`tfx-env`** kernel (`tensorflow-model-analysis`, `tensorflow-data-validation`, `tfx`).

In [23]:
import pathlib
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

import tensorflow_data_validation as tfdv
import tensorflow_model_analysis as tfma
from typing import cast

from google.protobuf.json_format import ParseDict
from google.protobuf.message import Message
from tensorflow_model_analysis.proto import config_pb2

# --- locate pipeline outputs (Airflow deployment or local repo clone) ---
def find_outputs_dir():
    roots = [
        pathlib.Path("/root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs"),
        pathlib.Path.cwd() / "tfx_airflow_runs" / "outputs",
        pathlib.Path.cwd().parent / "tfx_airflow_runs" / "outputs",
    ]
    for r in roots:
        r = r.resolve()
        if r.is_dir():
            return r
    raise FileNotFoundError(
        "No pipeline outputs found. Run the DAG first, or open the notebook from the repo / set path manually."
    )


def latest_version_dir(component_rel: str) -> pathlib.Path:
    """e.g. component_rel='StatisticsGen/statistics' -> .../statistics/<max_id>"""
    base = OUTPUTS / component_rel
    if not base.is_dir():
        raise FileNotFoundError(base)
    ids = [int(p.name) for p in base.iterdir() if p.is_dir() and p.name.isdigit()]
    if not ids:
        raise FileNotFoundError(f"No version dirs under {base}")
    return base / str(max(ids))


OUTPUTS = find_outputs_dir()
print("Using pipeline outputs:", OUTPUTS)

Using pipeline outputs: /root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs


## Step 1 — ExampleGen (artifact locations)

Train / eval TFRecord shards produced by `CsvExampleGen` (hash split 2:1).

In [15]:
eg = latest_version_dir("CsvExampleGen/examples")
train_uri = eg / "Split-train"
eval_uri = eg / "Split-eval"
print("ExampleGen version:", eg.name)
print("Train URI:", train_uri)
print("Eval URI:", eval_uri)
for name, u in [("train", train_uri), ("eval", eval_uri)]:
    if u.is_dir():
        n = len(list(u.glob("*")))
        print(f"  {name} shard files (~{n})")

ExampleGen version: 27
Train URI: /root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs/CsvExampleGen/examples/27/Split-train
Eval URI: /root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs/CsvExampleGen/examples/27/Split-eval
  train shard files (~1)
  eval shard files (~1)


## Step 2 — StatisticsGen (TFDV)

Interactive comparison of **train** vs **eval** feature statistics.

In [35]:
# StatisticsGen writes binary DatasetFeatureStatisticsList at Split-*/FeatureStats.pb
# (tfdv.load_statistics() expects a TFRecord or text proto file, not a directory.)
stats_root = latest_version_dir("StatisticsGen/statistics")
train_stats = tfdv.load_stats_binary(str(stats_root / "Split-train" / "FeatureStats.pb"))
eval_stats = tfdv.load_stats_binary(str(stats_root / "Split-eval" / "FeatureStats.pb"))
# TFDV 1.x: two protos + labels (dict API is newer only).
tfdv.visualize_statistics(
    train_stats, eval_stats, lhs_name="train", rhs_name="eval"
)

## Step 3 — SchemaGen (TFDV)

Inferred schema from training data.

In [4]:
schema_dir = latest_version_dir("SchemaGen/schema")
schema_path = schema_dir / "schema.pbtxt"
schema = tfdv.load_schema_text(str(schema_path))
tfdv.display_schema(schema)

,Type,Presence,Valency,Domain
Feature name,,,,
'Age',INT,required,,-
'CAEC',STRING,required,,'CAEC'
'CALC',STRING,required,,'CALC'
'CH2O',FLOAT,required,,-
'FAF',FLOAT,required,,-
'FAVC',STRING,required,,'FAVC'
'FCVC',FLOAT,required,,-
'Gender',STRING,required,,'Gender'
'Height',FLOAT,required,,-


,Values
Domain,
'CAEC',"'Always', 'Frequently', 'Sometimes', 'no'"
'CALC',"'Always', 'Frequently', 'Sometimes', 'no'"
'FAVC',"'no', 'yes'"
'Gender',"'Female', 'Male'"
'MTRANS',"'Automobile', 'Bike', 'Motorbike', 'Public_Transportation', 'Walking'"
'NObeyesdad',"'Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II'"
'SCC',"'no', 'yes'"
'SMOKE',"'no', 'yes'"
'family_history_with_overweight',"'no', 'yes'"


## Step 4 — ExampleValidator (TFDV)

Load anomaly artifacts per split (binary `SchemaDiff.pb` → `Anomalies`).

In [ ]:
from tensorflow_data_validation.utils.anomalies_util import load_anomalies_binary
from tensorflow_data_validation.utils.display_util import display_anomalies

val_root = latest_version_dir("ExampleValidator/anomalies")
for split in ("Split-train", "Split-eval"):
    p = val_root / split / "SchemaDiff.pb"
    if not p.is_file():
        print(split, ": no SchemaDiff.pb")
        continue
    anom = load_anomalies_binary(str(p))
    print("===", split, "===")
    display_anomalies(anom)

=== Split-train ===


=== Split-eval ===


## Step 5 — Transform (post-transform statistics)

Distribution of **transformed** features (after TFT `preprocessing_fn`).

In [38]:
post_dir = latest_version_dir("Transform/post_transform_stats")
post_stats = tfdv.load_stats_binary(str(post_dir / "FeatureStats.pb"))
tfdv.visualize_statistics(post_stats)

## Step 6 — Trainer (SavedModel)

Keras **SavedModel** produced by `obesity_tfx/trainer.py` (TFX `Trainer`). The servable lives under **`Format-Serving`** — same graph TFMA evaluates and `Pusher` can deploy when the model is **blessed**.

In [37]:
trainer_run = latest_version_dir("Trainer/model")
saved_model_dir = trainer_run / "Format-Serving"
print("Trainer artifact id:", trainer_run.name)
print("SavedModel (Format-Serving):", saved_model_dir.resolve())
if not saved_model_dir.is_dir():
    print("(Format-Serving not found yet — run the pipeline through Trainer.)")

Trainer artifact id: 32
SavedModel (Format-Serving): /root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs/Trainer/model/32/Format-Serving


## Step 8 — Evaluator (TFMA)

**Overall metrics**, **slices** (`Gender`, `Age_bucket` per pipeline `EvalConfig`), **time series** across **all** Evaluator artifact runs (when you have more than one pipeline execution), and **plots** when present.

Uses `tensorflow_model_analysis` (`load_eval_result`, `load_eval_results`, `tfma.view.render_*`).

In [36]:
_eval_base = OUTPUTS / "Evaluator" / "evaluation"
_ev_ids = sorted(
    int(p.name) for p in _eval_base.iterdir() if p.is_dir() and p.name.isdigit()
)
if not _ev_ids:
    raise FileNotFoundError(_eval_base)
eval_dirs = [_eval_base / str(i) for i in _ev_ids]
eval_dir = eval_dirs[-1]
print("TFMA evaluation directory (latest):", eval_dir)
if len(eval_dirs) > 1:
    print("Evaluator run ids (oldest → newest):", [p.name for p in eval_dirs])

eval_result = tfma.load_eval_result(str(eval_dir))

# Overall slicing metrics (all slices table)
tfma.view.render_slicing_metrics(eval_result)

TFMA evaluation directory (latest): /root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs/Evaluator/evaluation/33


SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

In [29]:
# Slice: Gender (as configured in tfx_pipeline EvalConfig)
_spec_gender = ParseDict({"feature_keys": ["Gender"]}, cast(Message, config_pb2.SlicingSpec()))
tfma.view.render_slicing_metrics(eval_result, slicing_spec=_spec_gender)

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Gender:1', 'metrics'…

In [30]:
# Slice: Age_bucket
_spec_age = ParseDict({"feature_keys": ["Age_bucket"]}, config_pb2.SlicingSpec())
tfma.view.render_slicing_metrics(eval_result, slicing_spec=_spec_age)

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Age_bucket:3', 'metr…

In [31]:
# Time series: one point per Evaluator artifact (re-run the DAG to add runs)
try:
    try:
        eval_results = tfma.load_eval_results(
            output_paths=[str(p) for p in eval_dirs]
        )
    except TypeError:
        eval_results = tfma.load_eval_results([str(p) for p in eval_dirs])
    tfma.view.render_time_series(eval_results)
except Exception as e:
    print("Time series viewer skipped:", e)

In [32]:
# Plots exported by TFMA (e.g. metric curves) — skipped if none
try:
    tfma.view.render_plot(eval_result)
except Exception as e:
    print("Plot viewer skipped:", e)

Plot viewer skipped: argument of type 'NoneType' is not iterable


In [43]:
# Summary: slice/metric materialization (structure varies by TFMA version)
sm = getattr(eval_result, "slicing_metrics", None)
if sm is not None:
    try:
        n = len(sm)
    except TypeError:
        n = "(not length-indexable)"
    print("Slicing metrics:", n)
else:
    print("No slicing_metrics on eval_result (check TFMA version).")

Slicing metrics: 9


## Step 9 — Pusher (blessed model)

When **Evaluator** metrics meet the blessing policy in `tfx_pipeline.py`, **Pusher** copies the blessed SavedModel to **`serving_model/`** at the **project root** (see `airflow_dags/obesity_dag.py`). Below: ML Metadata artifact under `outputs/Pusher/…` and the **filesystem** path used for serving / TensorFlow Serving.

In [44]:
pushed_run = latest_version_dir("Pusher/pushed_model")
print("Pusher artifact id:", pushed_run.name)
print("Pushed SavedModel root:", pushed_run.resolve())
_serving_root = OUTPUTS.parent.parent / "serving_model"
print("Filesystem serving copy (project serving_model/):", _serving_root.resolve())

Pusher artifact id: 42
Pushed SavedModel root: /root/airflow/dags/COMP315_Group8_project/tfx_airflow_runs/outputs/Pusher/pushed_model/42
Filesystem serving copy (project serving_model/): /root/airflow/dags/COMP315_Group8_project/serving_model
